# NLP Assignment 1
Text Preprocessing

**Name:** Ritweek Raj  
**Roll Number:** U24AI067  
**Language:** Hindi  

In [17]:
import re
import os
import pandas as pd
from datasets import load_dataset

In [18]:
# Load IndicCorpV2 for Hindi (using config indiccorp_v2 and split hin_Deva)
try:
    indic_dataset = load_dataset(
        "ai4bharat/IndicCorpV2",
        "indiccorp_v2",
        split="hin_Deva",
        streaming=True
    )
    print("IndicCorpV2 loaded successfully in streaming mode.")
except Exception as e:
    print("Error loading IndicCorpV2:", e)

IndicCorpV2 loaded successfully in streaming mode.


In [3]:
# Inspect dataset schema and sample
try:
    sample = next(iter(indic_dataset))
    print("IndicCorpV2 Schema / Keys:", list(sample.keys()))
    print("IndicCorpV2 Sample:", sample)
except Exception as e:
    print("Error inspecting IndicCorpV2 sample:", e)

IndicCorpV2 Schema / Keys: ['text']
IndicCorpV2 Sample: {'text': 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम'}


In [19]:
def sentence_tokenizer(text):
    """
    Splits text into sentences based on '.', '?', '!', and '।'.
    Protects URLs, emails, dates, and decimals from being split.
    """
    url_pattern = r'https?://[^\s!।]*[^\s!?।.,:;\'"`\(\)\[\]\{\}]'
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    date_pattern = r'\b\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}\b'
    decimal_pattern = r'\b\d+\.\d+\b'

    # Combine into a single regex for protected structures
    protected_regex = re.compile(
        f'({url_pattern}|{email_pattern}|{date_pattern}|{decimal_pattern})'
    )

    # Find all spans of protected structures
    protected_spans = []
    for match in protected_regex.finditer(text):
        protected_spans.append(match.span())

    # Find all potential sentence delimiters: ., ?, !, ।
    delimiter_regex = re.compile(r'[.?!।]')
    split_indices = []

    for match in delimiter_regex.finditer(text):
        start = match.start()
        # Check if this index falls inside any protected span
        inside_protected = False
        for p_start, p_end in protected_spans:
            if p_start <= start < p_end:
                inside_protected = True
                break
        if not inside_protected:
            split_indices.append(match.end())

    # Split the text based on split_indices
    sentences = []
    prev_idx = 0
    for idx in split_indices:
        sentence = text[prev_idx:idx].strip()
        if sentence:
            sentences.append(sentence)
        prev_idx = idx
    # Append the remaining text if any
    last_sentence = text[prev_idx:].strip()
    if last_sentence:
        sentences.append(last_sentence)

    return sentences

In [20]:
def word_tokenizer(sentence):
    """
    Splits a sentence into word tokens, preserving URLs, emails, dates, decimals, 
    Hindi words (excluding danda), English words, numbers, and splitting punctuation individual-by-individual.
    """
    url_pattern = r'https?://[^\s!।]*[^\s!?।.,:;\'"`\(\)\[\]\{\}]'
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    date_pattern = r'\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}'
    decimal_pattern = r'\d+\.\d+'
    hindi_pattern = r'[\u0900-\u0963\u0970-\u097F]+'
    english_pattern = r'[a-zA-Z]+'
    number_pattern = r'[\d\u0966-\u096f]+'
    punctuation_pattern = r'[^\w\s\u0900-\u097F]|।|॥'

    # Fallback to \S guarantees no character is dropped
    token_pattern = re.compile(
        f'({url_pattern}|{email_pattern}|{date_pattern}|{decimal_pattern}|{hindi_pattern}|{english_pattern}|{number_pattern}|{punctuation_pattern}|\\S)'
    )

    tokens = [match.group() for match in token_pattern.finditer(sentence)]
    return tokens

In [21]:
# Test cases to verify the tokenizer manually
test_texts = [
    "Hello. 3.14 is a decimal.",
    "abc@gmail.com is my email.",
    "https://google.com is a URL.",
    "12/05/2025 is a date.",
    "भारत महान है।"
]

for text in test_texts:
    print(f"Raw text: {repr(text)}")
    sentences = sentence_tokenizer(text)
    print(f"Sentences: {sentences}")
    for sentence in sentences:
        words = word_tokenizer(sentence)
        print(f"  Words: {words}")
    print("-" * 45)

Raw text: 'Hello. 3.14 is a decimal.'
Sentences: ['Hello.', '3.14 is a decimal.']
  Words: ['Hello', '.']
  Words: ['3.14', 'is', 'a', 'decimal', '.']
---------------------------------------------
Raw text: 'abc@gmail.com is my email.'
Sentences: ['abc@gmail.com is my email.']
  Words: ['abc@gmail.com', 'is', 'my', 'email', '.']
---------------------------------------------
Raw text: 'https://google.com is a URL.'
Sentences: ['https://google.com is a URL.']
  Words: ['https://google.com', 'is', 'a', 'URL', '.']
---------------------------------------------
Raw text: '12/05/2025 is a date.'
Sentences: ['12/05/2025 is a date.']
  Words: ['12/05/2025', 'is', 'a', 'date', '.']
---------------------------------------------
Raw text: 'भारत महान है।'
Sentences: ['भारत महान है।']
  Words: ['भारत', 'महान', 'है', '।']
---------------------------------------------


In [22]:
def tokenize_corpus(dataset, max_sentences=50000):
    """
    Iterates through dataset paragraphs, tokenizes them into sentences and words, 
    and joins the word tokens with a single space.
    """
    tokenized_sentences = []
    count = 0
    
    for sample in dataset:
        text = sample.get("text", "")
        if not text:
            continue
        
        sentences = sentence_tokenizer(text)
        for s in sentences:
            words = word_tokenizer(s)
            # Join tokens with space
            tokenized_s = " ".join(words)
            tokenized_sentences.append(tokenized_s)
            count += 1
            if max_sentences and count >= max_sentences:
                break
        
        if max_sentences and count >= max_sentences:
            break
            
    return tokenized_sentences

In [23]:
def save_parquet(tokenized_sentences, filename):
    """
    Saves the list of tokenized sentences into a Parquet file with a single column 'sentence'.
    """
    df = pd.DataFrame({"sentence": tokenized_sentences})
    df.to_parquet(filename, index=False)
    print(f"Saved {len(df)} sentences to {filename}")

In [24]:
def corpus_statistics(tokenized_sentences):
    """
    Calculates required statistics for the tokenized corpus:
    - Total Sentences
    - Total Words
    - Total Characters (excluding spaces between tokens)
    - Average Sentence Length
    - Average Word Length
    - Type Token Ratio (TTR)
    """
    total_sentences = len(tokenized_sentences)
    
    all_words = []
    for s in tokenized_sentences:
        all_words.extend(s.split())
        
    total_words = len(all_words)
    total_chars = sum(len(w) for w in all_words)
    
    avg_sentence_len = total_words / total_sentences if total_sentences > 0 else 0
    avg_word_len = total_chars / total_words if total_words > 0 else 0
    
    unique_words = set(all_words)
    ttr = len(unique_words) / total_words if total_words > 0 else 0
    
    stats = {
        "Total Sentences": total_sentences,
        "Total Words": total_words,
        "Total Characters": total_chars,
        "Average Sentence Length": avg_sentence_len,
        "Average Word Length": avg_word_len,
        "Type Token Ratio": ttr
    }
    return stats

In [25]:
def save_statistics(stats, filename):
    """
    Writes the corpus statistics to a text file.
    """
    with open(filename, "w") as f:
        for k, v in stats.items():
            f.write(f"{k}: {v}\n")
    print(f"Saved statistics to {filename}")

In [26]:
def process_dataset(dataset, name, limit=50000):
    """
    Pipeline: Load -> Tokenize -> Save Parquet -> Statistics -> Save Statistics
    """
    print(f"=== Processing {name} Dataset ===")
    tokenized_sentences = tokenize_corpus(dataset, max_sentences=limit)
    
    parquet_file = f"{name.lower()}_tokenized.parquet"
    save_parquet(tokenized_sentences, parquet_file)
    
    stats = corpus_statistics(tokenized_sentences)
    for k, v in stats.items():
        print(f"{k}: {v}")
        
    stats_file = f"{name.lower()}_stats.txt"
    save_statistics(stats, stats_file)
    print()
    return tokenized_sentences, stats

In [27]:
# Run tokenization pipeline for IndicCorpV2 (Hindi)
indic_tokenized, indic_stats = process_dataset(indic_dataset, "indic", limit=50000)

=== Processing indic Dataset ===
Saved 50000 sentences to indic_tokenized.parquet
Total Sentences: 50000
Total Words: 919334
Total Characters: 3403766
Average Sentence Length: 18.38668
Average Word Length: 3.7024258865657096
Type Token Ratio: 0.05345500112037627
Saved statistics to indic_stats.txt



In [ ]:
# Load OSCAR-2301 for Hindi
try:
    oscar_dataset = load_dataset(
        "oscar-corpus/OSCAR-2301",
        "hi",
        split="train",
        streaming=False
    )
    print("OSCAR-2301 loaded successfully.")
except Exception as e:
    print("Error loading OSCAR-2301:", e)
    print("OSCAR-2301 is a gated repository. If you haven't done so, please login via browser or 'huggingface-cli login' and accept the license on HF: https://huggingface.co/datasets/oscar-corpus/OSCAR-2301")
    oscar_dataset = None

Error loading OSCAR-2301: 403 Client Error. (Request ID: Root=1-6a74c3e4-6a59559c3e147abd292d4c3d;0b4d9856-98c5-42ad-8c00-bc8c1ae2057a)

Cannot access gated repo for url https://huggingface.co/datasets/oscar-corpus/OSCAR-2301/resolve/main/OSCAR-2301.py.
Access to dataset oscar-corpus/OSCAR-2301 is restricted and you are not in the authorized list. Visit https://huggingface.co/datasets/oscar-corpus/OSCAR-2301 to ask for access.
OSCAR-2301 is a gated repository. If you haven't done so, please login via browser or 'huggingface-cli login' and accept the license on HF: https://huggingface.co/datasets/oscar-corpus/OSCAR-2301


In [ ]:
# Run tokenization pipeline for OSCAR-2301 (Hindi) if loaded
if oscar_dataset is not None:
    oscar_tokenized, oscar_stats = process_dataset(oscar_dataset, "oscar", limit=50000)
else:
    print("Skipping OSCAR-2301 processing since the dataset could not be loaded.")

NameError: name 'oscar_dataset' is not defined